# Forward-Risk-Manager End-to-End Runbook

This single notebook runs the repo workflow from start to finish:
- optional environment install
- graph build
- strict two-stage training (encoder then critic)
- benchmark + sanity checks
- hyperparameter sweep + summary + plots
- scenario book + stress test + hallucination calibration
- optional goodness backtest
- final artifact inventory

All outputs are redirected to an isolated run folder under `runs/experiments/`.

In [6]:
import csv
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from collections import deque
from pathlib import Path

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

import pandas as pd
import torch


def q(value) -> str:
    return shlex.quote(str(value))


def run(cmd: str, allow_fail: bool = False, tail_lines: int = 200) -> bool:
    print("\n" + "=" * 110)
    print(cmd)
    print("=" * 110)
    t0 = time.time()

    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

    logs_dir = Path("runs/experiments/_e2e_logs")
    logs_dir.mkdir(parents=True, exist_ok=True)
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", cmd).strip("_")[:120] or "command"
    log_path = logs_dir / f"{int(time.time())}_{safe_name}.log"

    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None

    tail = deque(maxlen=max(40, int(tail_lines)))
    with log_path.open("w", encoding="utf-8") as lf:
        for line in proc.stdout:
            print(line, end="")
            lf.write(line)
            tail.append(line.rstrip("\n"))
    rc = proc.wait()

    elapsed = time.time() - t0
    print(f"\ncompleted in {elapsed:.2f}s | log: {log_path}")
    if rc != 0:
        if tail:
            print("---- command output tail ----")
            for ln in tail:
                print(ln)
            print("---- end tail ----")
        msg = f"command failed ({rc}): {cmd}"
        if allow_fail:
            print("WARNING:", msg)
            return False
        raise RuntimeError(msg)
    return True


def _toml_value_literal(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, int) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, float):
        return repr(float(value))
    if isinstance(value, list):
        return "[" + ", ".join(_toml_value_literal(v) for v in value) + "]"
    return json.dumps(str(value))


def write_runtime_config(base_config_path: str, runtime_config_path: str, section_overrides: dict) -> str:
    src = Path(base_config_path)
    dst = Path(runtime_config_path)
    lines = src.read_text().splitlines()

    section_re = re.compile(r"^\s*\[([^\]]+)\]\s*$")
    key_res = {
        sec: {k: re.compile(rf"^\s*{re.escape(k)}\s*=") for k in kv}
        for sec, kv in section_overrides.items()
    }
    replaced = {(sec, key): False for sec, kv in section_overrides.items() for key in kv}
    seen_sections = set()

    out = []
    current_section = None

    def flush_missing(section_name):
        if section_name not in section_overrides:
            return
        for key, value in section_overrides[section_name].items():
            if not replaced[(section_name, key)]:
                out.append(f"{key} = {_toml_value_literal(value)}")
                replaced[(section_name, key)] = True

    for line in lines:
        m = section_re.match(line)
        if m:
            flush_missing(current_section)
            current_section = m.group(1).strip()
            seen_sections.add(current_section)
            out.append(line)
            continue

        updated = False
        if current_section in section_overrides:
            for key, pattern in key_res[current_section].items():
                if pattern.match(line):
                    out.append(f"{key} = {_toml_value_literal(section_overrides[current_section][key])}")
                    replaced[(current_section, key)] = True
                    updated = True
                    break

        if not updated:
            out.append(line)

    flush_missing(current_section)

    for section_name, overrides in section_overrides.items():
        if section_name in seen_sections:
            continue
        if out and out[-1].strip() != "":
            out.append("")
        out.append(f"[{section_name}]")
        for key, value in overrides.items():
            out.append(f"{key} = {_toml_value_literal(value)}")

    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text("\n".join(out) + "\n")
    return str(dst)


In [7]:
# Controls
RUN_PROFILE = "full"  # "full" or "fast"
INSTALL_DEPS = ("google.colab" in sys.modules)
RUN_SWEEP_PROMOTION = True
RUN_OPTIONAL_BACKTEST = True
FORCE_REBUILD_GRAPHS = False
AUTO_PREP_DATA = True

# Optional explicit overrides. Leave empty to auto-detect.
DATA_PRICES_PATH = ""
DATA_CONSTITUENTS_PATH = ""
DATA_FUNDAMENTALS_PATH = ""

assert RUN_PROFILE in {"full", "fast"}

IN_COLAB = "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)
print("INSTALL_DEPS:", INSTALL_DEPS)

# Colab runtime and GPU sanity checks (aligned with notebooks/colab_setup.ipynb)
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    print("python:", sys.version)
    print("python_executable:", sys.executable)
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    print("cuda_version:", torch.version.cuda)

    try:
        print(subprocess.check_output(["nvidia-smi", "-L"], text=True))
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. In Colab, switch Runtime -> GPU and rerun.")

    gpu_name = torch.cuda.get_device_name(0)
    print("gpu:", gpu_name)
    if "T4" not in gpu_name.upper():
        print("WARNING: GPU is not T4. Notebook still works; timings will differ.")

    # Throughput-friendly defaults for CUDA/T4.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("FRM_REPO_DIR", "").strip()
    if env_repo and Path(env_repo).exists() and (Path(env_repo) / "configs/default.toml").exists():
        return Path(env_repo)

    candidates = [
        Path.cwd(),
        Path("/content/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/forward-risk-manager"),
    ]

    if IN_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.exists():
            candidates.extend(
                sorted(
                    p for p in drive_root.glob("*Forward*Risk*Manager*")
                    if p.is_dir()
                )
            )

    for p in candidates:
        if (p / "configs/default.toml").exists():
            return p

    raise FileNotFoundError(
        "Could not find repo root containing configs/default.toml. "
        "Set FRM_REPO_DIR or update candidate paths in this cell."
    )


REPO_DIR = _resolve_repo_dir()
os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
print("cwd:", Path.cwd())

if Path(".git").exists():
    status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
    if status.returncode == 0:
        out = status.stdout.strip()
        print(out[:4000] if out else "Git repo detected. Working tree clean.")
    else:
        msg = (status.stderr or status.stdout or "").strip()
        print(f"`git status --short` failed with exit code {status.returncode}.")
        if msg:
            print(msg[:4000])
        if "dubious ownership" in msg.lower():
            print('Fix: run `!git config --global --add safe.directory "$PWD"` and retry.')

venv_py = Path(".venv/bin/python")
PYTHON_EXE = venv_py if venv_py.exists() else Path(sys.executable)
print("python exe for commands:", PYTHON_EXE)

BASE_CONFIG = Path("configs/default.toml")
assert BASE_CONFIG.exists(), f"Missing config: {BASE_CONFIG}"

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_ROOT = Path("runs/experiments") / f"e2e_runbook_{RUN_ID}"
METRICS_DIR = RUN_ROOT / "metrics"
PLOTS_DIR = RUN_ROOT / "plots"
LOGS_DIR = RUN_ROOT / "logs"
MODELS_DIR = RUN_ROOT / "models"
DIAG_DIR = RUN_ROOT / "diagnostics"
DATA_DIR = RUN_ROOT / "data"

for p in [METRICS_DIR, PLOTS_DIR, LOGS_DIR, MODELS_DIR, DIAG_DIR, DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

runtime_config = RUN_ROOT / "runtime_config.toml"

graphs_pt = DATA_DIR / "graphs.pt"
model_ckpt = MODELS_DIR / "encoder.pt"
encoder_ckpt = MODELS_DIR / "encoder.pt"
critic_ckpt = MODELS_DIR / "critic.pt"

benchmark_csv = METRICS_DIR / "benchmark.csv"
benchmark_plot = PLOTS_DIR / "benchmark_speed_sep.png"
benchmark_bar = PLOTS_DIR / "benchmark_bar.png"

sweep_csv = METRICS_DIR / "ff_sweep.csv"
sweep_summary = LOGS_DIR / "ff_sweep_summary.txt"
sweep_plot = PLOTS_DIR / "ff_sweep_tradeoff.png"
sweep_pareto = PLOTS_DIR / "ff_sweep_pareto.png"

scenario_csv = METRICS_DIR / "scenario_book.csv"
scenario_diag = DIAG_DIR / "scenario_constraint_diagnostics.csv"
stress_csv = METRICS_DIR / "stress_test_report.csv"
stress_plot = PLOTS_DIR / "stress_test_report.png"
calibration_json = DIAG_DIR / "hallucination_calibration.json"
calibration_by_ticker = DIAG_DIR / "hallucination_calibration_by_ticker.csv"

goodness_csv = DIAG_DIR / "goodness_backtest.csv"
goodness_quantiles = DIAG_DIR / "goodness_quantiles.csv"
goodness_plot = PLOTS_DIR / "goodness_scatter.png"
goodness_events = DIAG_DIR / "goodness_events.csv"
goodness_strategy = DIAG_DIR / "goodness_strategy_metrics.csv"
goodness_timeline = PLOTS_DIR / "goodness_timeline.png"


def _find_tidy_sources() -> tuple[str, Path, Path, Path | None]:
    if DATA_PRICES_PATH or DATA_CONSTITUENTS_PATH:
        prices = Path(DATA_PRICES_PATH) if DATA_PRICES_PATH else None
        consts = Path(DATA_CONSTITUENTS_PATH) if DATA_CONSTITUENTS_PATH else None
        if prices is None or consts is None:
            raise ValueError("Set both DATA_PRICES_PATH and DATA_CONSTITUENTS_PATH, or leave both empty.")
        if not prices.exists() or not consts.exists():
            raise FileNotFoundError(
                f"Explicit data path missing: prices={prices} exists={prices.exists()} | "
                f"constituents={consts} exists={consts.exists()}"
            )
        fund = Path(DATA_FUNDAMENTALS_PATH) if DATA_FUNDAMENTALS_PATH else None
        if fund is not None and not fund.exists():
            print(f"WARNING: DATA_FUNDAMENTALS_PATH missing, ignoring: {fund}")
            fund = None
        return "manual", prices, consts, fund

    candidate_pairs = [
        (
            "processed_long",
            Path("data/processed_long/prices.csv"),
            Path("data/processed_long/constituents.csv"),
            Path("data/processed_long/fundamentals.csv"),
        ),
        (
            "processed",
            Path("data/processed/prices.csv"),
            Path("data/processed/constituents.csv"),
            Path("data/processed/fundamentals.csv"),
        ),
        (
            "raw_merged",
            Path("data/raw_merged/prices.csv"),
            Path("data/raw_merged/constituents.csv"),
            Path("data/raw_merged/fundamentals.csv"),
        ),
    ]

    # Add recursive fallback discovery under data/.
    data_root = Path("data")
    if data_root.exists():
        for price_path in sorted(data_root.rglob("prices.csv")):
            const_path = price_path.with_name("constituents.csv")
            fund_path = price_path.with_name("fundamentals.csv")
            if const_path.exists():
                candidate_pairs.append((f"discovered:{price_path.parent}", price_path, const_path, fund_path))

    for label, prices, consts, fund in candidate_pairs:
        if prices.exists() and consts.exists():
            return label, prices, consts, fund if fund.exists() else None

    # Optional auto-prepare path for year-bucketed raw exports.
    if AUTO_PREP_DATA and Path("data/raw").exists():
        print("No tidy prices/constituents found. Attempting merge_raw_years auto-prepare...")
        run(
            f"{q(PYTHON_EXE)} scripts/merge_raw_years.py --raw-root data/raw --out-dir data/raw_merged",
            allow_fail=False,
        )
        prices = Path("data/raw_merged/prices.csv")
        consts = Path("data/raw_merged/constituents.csv")
        fund = Path("data/raw_merged/fundamentals.csv")
        if prices.exists() and consts.exists():
            return "raw_merged(auto_prepared)", prices, consts, fund if fund.exists() else None

    raise FileNotFoundError(
        "Could not find tidy inputs for graph building (prices.csv + constituents.csv).\n"
        "Provide DATA_PRICES_PATH/DATA_CONSTITUENTS_PATH in this cell, or prepare data via one of:\n"
        "1) python scripts/qc_export_to_tidy.py --prices ... --constituents ... --out-dir data/processed\n"
        "2) python scripts/merge_raw_years.py --raw-root data/raw --out-dir data/raw_merged"
    )


data_source_label, build_prices_path, build_constituents_path, build_fundamentals_path = _find_tidy_sources()
print("data source:", data_source_label)
print("build prices:", build_prices_path)
print("build constituents:", build_constituents_path)
print("build fundamentals:", build_fundamentals_path if build_fundamentals_path else "<none>")

overrides = {
    "build_graphs": {
        "prices": str(build_prices_path),
        "constituents": str(build_constituents_path),
        "out": str(graphs_pt),
    },
    "train": {
        "graphs": str(graphs_pt),
        "save_model": str(model_ckpt),
        "save_encoder": str(encoder_ckpt),
        "save_critic": str(critic_ckpt),
        "log_csv": str(METRICS_DIR / "ff_train.csv"),
        "plot_path": str(PLOTS_DIR / "ff_train.png"),
        "strict_component_split": True,
    },
    "benchmark": {
        "out_csv": str(benchmark_csv),
        "plot_path": str(benchmark_plot),
        "bar_plot_path": str(benchmark_bar),
    },
    "sweep": {
        "out_csv": str(sweep_csv),
        "top_k": 10,
    },
    "scenario_book": {
        "critic_model": str(critic_ckpt),
        "out": str(scenario_csv),
        "diag_out": str(scenario_diag),
    },
    "encoder": {
        "neg_mode": "self_contrastive",
        "strict_component_split": True,
        "freeze_critic": True,
        "save_encoder": str(encoder_ckpt),
    },
    "critic": {
        "neg_mode": "time_flip+noise",
        "strict_component_split": True,
        "freeze_encoder": True,
        "encoder_checkpoint_in": str(encoder_ckpt),
        "save_critic": str(critic_ckpt),
    },
}

if build_fundamentals_path is not None:
    overrides["build_graphs"]["fundamentals"] = str(build_fundamentals_path)

if RUN_PROFILE == "fast":
    overrides["train"].update({
        "epochs": 25,
        "batch_size": 16,
    })
    overrides["benchmark"].update({
        "epochs": 3,
        "batch_size": 16,
    })
    overrides["sweep"].update({
        "epochs": 2,
        "batch_size": 16,
        "goodness_temp": [0.2],
        "goodness_target": [1.5],
        "neg_mix_end": [0.5],
        "hall_steps": [1],
        "hall_lr": [0.02],
        "hall_node_fraction": [0.2],
        "top_k": 3,
    })
    overrides["scenario_book"].update({
        "num_scenarios": 12,
        "max_adapt_steps": 8,
    })
    overrides["encoder"].update({
        "epochs": 12,
        "batch_size": 16,
    })
    overrides["critic"].update({
        "epochs": 12,
        "batch_size": 16,
    })

write_runtime_config(str(BASE_CONFIG), str(runtime_config), overrides)
print("runtime config:", runtime_config)
print("run root:", RUN_ROOT)




IN_COLAB: True
INSTALL_DEPS: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
python_executable: /usr/bin/python3
torch: 2.9.0+cu128
cuda_available: True
cuda_version: 12.8
GPU 0: Tesla T4 (UUID: GPU-676ce11b-5f2b-53f2-e376-1e51a5d6cf82)

gpu: Tesla T4
repo: /content/drive/My Drive/Forward-Risk-Manager
cwd: /content/drive/My Drive/Forward-Risk-Manager
M README.md
 M configs/baseline.toml
 M configs/default.toml
 M configs/train_long_constituents.toml
 M notebooks/colab_setup.ipynb
 M scripts/benchmark_training.py
 M scripts/dual_score_report.py
 M scripts/ff_sweep.py
 M scripts/hallucination_calibration.py
 M scripts/promote_sweep_best.py
 M scripts/publish_run.py
 M scripts/qc_export_to_tidy.py
 M scripts/sanity_checks.py
 M scripts/scenario_book.py
 M scripts/stress_test_report.py
 M scripts/train_ff_gnn.py
 M src/forward_risk_manager.egg-inf

In [8]:
import importlib.util

required_modules = [
    "torch",
    "torch_geometric",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
    "joblib",
]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]

need_install = bool(INSTALL_DEPS) or bool(missing)
if need_install:
    if missing and not INSTALL_DEPS:
        print("Missing modules detected:", missing)
        print("Auto-installing requirements to satisfy missing dependencies.")
    run(f"{q(PYTHON_EXE)} -m pip install --upgrade pip")
    run(f"{q(PYTHON_EXE)} -m pip install -r requirements.txt")
    run(f"{q(PYTHON_EXE)} -m pip install -e .")
else:
    print("Dependencies already present. Skipping install.")

# Verify critical import before starting expensive pipeline steps.
import torch_geometric  # noqa: F401
print("torch_geometric import OK")




/usr/bin/python3 -m pip install --upgrade pip

completed in 1.64s | log: runs/experiments/_e2e_logs/1771016138_usr_bin_python3_-m_pip_install_--upgrade_pip.log

/usr/bin/python3 -m pip install -r requirements.txt

completed in 1.56s | log: runs/experiments/_e2e_logs/1771016139_usr_bin_python3_-m_pip_install_-r_requirements.txt.log

/usr/bin/python3 -m pip install -e .
Obtaining file:///content/drive/MyDrive/Forward-Risk-Manager
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for forward-risk-manager (pyproject.toml): started


## Step 1: Build Graphs

In [9]:
if FORCE_REBUILD_GRAPHS or not graphs_pt.exists():
    run(f"{q(PYTHON_EXE)} scripts/build_graphs.py --config {q(runtime_config)}")
else:
    print("Graphs already exist, skipping rebuild:", graphs_pt)

assert graphs_pt.exists(), f"Missing graphs payload: {graphs_pt}"

try:
    payload = torch.load(graphs_pt, map_location="cpu", weights_only=False)
except TypeError:
    payload = torch.load(graphs_pt, map_location="cpu")

graphs = payload["graphs"] if isinstance(payload, dict) else payload
dates = payload.get("dates", []) if isinstance(payload, dict) else []
tickers = payload.get("tickers", []) if isinstance(payload, dict) else []
print("num_graphs:", len(graphs))
print("num_dates:", len(dates))
print("has_tickers:", bool(tickers))
if graphs:
    g0 = graphs[0]
    print("sample graph nodes:", int(g0.num_nodes), "features:", int(g0.x.shape[1]))


/usr/bin/python3 scripts/build_graphs.py --config runs/experiments/e2e_runbook_20260213_205538/runtime_config.toml

Building graphs: 100%|██████████| 6527/6527 [02:39<00:00, 40.90win/s]
Wrote runs/experiments/e2e_runbook_20260213_205538/data/graphs.pt with 2064 graphs
Date range: 2011-06-01 -> 2026-02-06 | windows: 6527 | built: 2064 | skipped_lag=0, skipped: members=4463, cols=0, min_nodes=0, no_edges=0

completed in 188.88s | log: runs/experiments/_e2e_logs/1771016148_usr_bin_python3_scripts_build_graphs.py_--config_runs_experiments_e2e_runbook_20260213_205538_runtime_config.toml.log
num_graphs: 2064
num_dates: 2064
has_tickers: True
sample graph nodes: 319 features: 25


## Step 2: Two-Stage Training (Encoder + Critic)

In [10]:
run(
    f"{q(PYTHON_EXE)} scripts/train_two_stage.py "
    f"--config {q(runtime_config)} "
    f"--encoder-out {q(encoder_ckpt)} "
    f"--critic-out {q(critic_ckpt)} "
    f"--critic-neg-mode time_flip+noise"
)

assert encoder_ckpt.exists(), f"Missing encoder checkpoint: {encoder_ckpt}"
assert critic_ckpt.exists(), f"Missing critic checkpoint: {critic_ckpt}"

# Keep save_model aligned for scripts that load train.save_model (e.g., goodness_backtest).
if not model_ckpt.exists() and encoder_ckpt.exists():
    shutil.copy2(encoder_ckpt, model_ckpt)

print("encoder bytes:", encoder_ckpt.stat().st_size)
print("critic bytes:", critic_ckpt.stat().st_size)
print("model bytes:", model_ckpt.stat().st_size if model_ckpt.exists() else 0)


/usr/bin/python3 scripts/train_two_stage.py --config runs/experiments/e2e_runbook_20260213_205538/runtime_config.toml --encoder-out runs/experiments/e2e_runbook_20260213_205538/models/encoder.pt --critic-out runs/experiments/e2e_runbook_20260213_205538/models/critic.pt --critic-neg-mode time_flip+noise
adaptive_goodness_target disabled for self_contrastive mode.
device request: auto
device: cuda
torch: 2.9.0+cu128
cuda_available: True
cuda_version: 12.8
mps_built: False
mps_available: False
cuda_device_name: Tesla T4
neg_mode: self_contrastive | batch_size: 32 | loader_workers: 0
ff_mode: layerwise=False, blockwise=False, block_size=2, multiscale=True
energy_penalty: 0.0001 (mode=last)
risk_head: ticker=AUTO horizons=[21] weight=0.1 type=huber std=True max_abs_logret=0.5
self_contrastive_temp: 0.2
self_contrastive_view: mode=shuffle+noise, noise_std=0.05
self_contrastive_energy_penalty_scale: 0.0
distance_forward: weight=0.02, margin=0.15
amp: enabled (float16)
torch threads: 2 | inte

## Step 3: Benchmark + Sanity Checks

In [11]:
run(f"{q(PYTHON_EXE)} scripts/benchmark_training.py --config {q(runtime_config)}")
sanity_ok = run(
    f"{q(PYTHON_EXE)} scripts/sanity_checks.py --benchmark-csv {q(benchmark_csv)}",
    allow_fail=True,
)
print("sanity_checks_passed:", sanity_ok)

assert benchmark_csv.exists(), f"Missing benchmark CSV: {benchmark_csv}"
benchmark_df = pd.read_csv(benchmark_csv)

required_cols = ["mode", "eval_objective", "objective_track", "primary_eval_metric_name", "primary_eval_metric"]
missing = [c for c in required_cols if c not in benchmark_df.columns]
assert not missing, f"Benchmark missing columns: {missing}"

print("benchmark rows:", len(benchmark_df))
print("objective_track counts:\n", benchmark_df["objective_track"].astype(str).value_counts(dropna=False))

view_cols = [c for c in [
    "mode", "row_type", "eval_objective", "objective_track",
    "eval_sep", "eval_sc_gap", "eval_auroc", "graphs_per_s"
] if c in benchmark_df.columns]
print(benchmark_df[view_cols].head(20).to_string(index=False))


/usr/bin/python3 scripts/benchmark_training.py --config runs/experiments/e2e_runbook_20260213_205538/runtime_config.toml
econ ticker: requested=AUTO effective=BWC source=auto_max_rows rows=10439

Benchmark: 100%|██████████| 5/5 [00:07<00:00,  1.59s/epoch]
calibrated goodness_target=1.9936 (train-cal acc=0.5000)

Benchmark: 100%|██████████| 5/5 [00:05<00:00,  1.13s/epoch]
calibrated goodness_target=5.9835 (train-cal acc=0.5000)

Benchmark: 100%|██████████| 5/5 [00:05<00:00,  1.10s/epoch]
Wrote runs/experiments/e2e_runbook_20260213_205538/metrics/benchmark.csv
{'avg_epoch_s': 1.4627413552499888, 'graphs_per_s': 1128.70484128883, 'goodness_target_eval': 1.9936376810073853, 'target_cal_acc': 0.5000000149011612, 'neg_mode_effective': 'mix', 'eval_neg_mode_effective': 'mix', 'eval_objective': 'ff', 'eval_g_pos': 5.30052610544058, 'eval_g_neg': 5.3005260687607985, 'eval_sep': 3.66797818784903e-08, 'eval_acc': 0.5, 'eval_auroc': 0.5000381077452527, 'eval_auprc': 0.5041524911839114, 'eval_brie

## Step 4: Sweep + Summary + Tradeoff Plots

In [12]:
run(f"{q(PYTHON_EXE)} scripts/ff_sweep.py --config {q(runtime_config)}")
run(f"{q(PYTHON_EXE)} scripts/ff_sweep_summary.py --csv {q(sweep_csv)} --out {q(sweep_summary)}")
run(f"{q(PYTHON_EXE)} scripts/plot_ff_sweep.py --csv {q(sweep_csv)} --out {q(sweep_plot)} --pareto-out {q(sweep_pareto)}")

if RUN_SWEEP_PROMOTION:
    run(f"{q(PYTHON_EXE)} scripts/promote_sweep_best.py --config {q(runtime_config)} --csv {q(sweep_csv)} --top-k 5")
else:
    print("Skipping promote_sweep_best (RUN_SWEEP_PROMOTION=False)")

assert sweep_csv.exists(), f"Missing sweep CSV: {sweep_csv}"
sweep_df = pd.read_csv(sweep_csv)
print("sweep rows:", len(sweep_df))
print("objective_track counts:\n", sweep_df["objective_track"].astype(str).value_counts(dropna=False))

sort_col = "rank_value" if "rank_value" in sweep_df.columns else "primary_eval_metric"
preview_cols = [c for c in [
    "mode", "eval_objective", "objective_track", "rank_metric", "rank_value",
    "primary_eval_metric_name", "primary_eval_metric", "graphs_per_s"
] if c in sweep_df.columns]
print(sweep_df.sort_values(sort_col, ascending=False).head(10)[preview_cols].to_string(index=False))


/usr/bin/python3 scripts/ff_sweep.py --config runs/experiments/e2e_runbook_20260213_205538/runtime_config.toml
econ ticker: requested=AUTO effective=BWC source=auto_max_rows rows=10439

Sweep: 100%|██████████| 256/256 [32:16<00:00,  7.57s/trial]
Wrote runs/experiments/e2e_runbook_20260213_205538/metrics/ff_sweep.csv
Best by rank_value (econ_sharpe_uplift): {'avg_epoch_s': 1.4804860395001924, 'graphs_per_s': 1115.2731333784004, 'neg_mode_effective': 'mix', 'eval_neg_mode_effective': 'mix', 'eval_objective': 'ff', 'eval_g_pos': 2.322385549545288, 'eval_g_neg': 2.3223855862250695, 'eval_sep': -3.667978143440109e-08, 'eval_acc': 0.5, 'eval_auroc': 0.4999648236197668, 'eval_auprc': 0.5043413829015801, 'eval_brier': 0.2970347127429237, 'eval_ece': 0.1616088532130773, 'econ_num_days': 412.0, 'econ_bh_total_return': 0.8262087285870563, 'econ_bh_ann_return': 0.44536468016318165, 'econ_bh_ann_vol': 0.3067036272038445, 'econ_bh_sharpe': 1.452101118671084, 'econ_bh_max_drawdown': -0.3185394340524

## Step 5: Scenario Book + Stress + Calibration

In [13]:
run(
    f"{q(PYTHON_EXE)} scripts/scenario_book.py "
    f"--config {q(runtime_config)} "
    f"--critic-model {q(critic_ckpt)} "
    f"--out {q(scenario_csv)} "
    f"--diag-out {q(scenario_diag)}"
)
run(
    f"{q(PYTHON_EXE)} scripts/stress_test_report.py "
    f"--csv {q(scenario_csv)} "
    f"--out-csv {q(stress_csv)} "
    f"--out-plot {q(stress_plot)}"
)
run(
    f"{q(PYTHON_EXE)} scripts/hallucination_calibration.py "
    f"--csv {q(scenario_csv)} "
    f"--out {q(calibration_json)} "
    f"--out-by-ticker {q(calibration_by_ticker)}"
)

scenario_df = pd.read_csv(scenario_csv)
required_meta = [
    "objective_track", "energy_component", "component_split_mode",
    "encoder_checkpoint", "critic_checkpoint", "train_neg_mode",
]
missing = [c for c in required_meta if c not in scenario_df.columns]
assert not missing, f"Scenario CSV missing metadata columns: {missing}"

assert scenario_df["objective_track"].astype(str).str.lower().eq("critic").all(), "Scenario rows should be critic-tracked"
assert scenario_df["energy_component"].astype(str).str.lower().eq("critic_energy").all(), "Scenario energy should be critic_energy"

print("scenario rows:", len(scenario_df))
meta_view = [c for c in [
    "scenario_id", "graph_index", "date", "target_ticker", "ticker", "series",
    "objective_track", "energy_component", "component_split_mode"
] if c in scenario_df.columns]
print(scenario_df[meta_view].head(20).to_string(index=False))

stress_df = pd.read_csv(stress_csv)
print("stress rows:", len(stress_df))
print(stress_df.head(20).to_string(index=False))

with Path(calibration_json).open() as f:
    calib = json.load(f)
print("calibration summary:")
print({k: calib.get(k) for k in ["num_pairs", "num_points", "num_tickers", "corr_real_hall", "mae", "js_divergence", "tail_ratio_p99"]})


/usr/bin/python3 scripts/scenario_book.py --config runs/experiments/e2e_runbook_20260213_205538/runtime_config.toml --critic-model runs/experiments/e2e_runbook_20260213_205538/models/critic.pt --out runs/experiments/e2e_runbook_20260213_205538/metrics/scenario_book.csv --diag-out runs/experiments/e2e_runbook_20260213_205538/diagnostics/scenario_constraint_diagnostics.csv
scenario target_ticker auto-resolved to AFG
adaptive attempt 1/40 | constraint_weight=20.000 | hall_steps=4 | hall_lr=0.0300 | hall_l2=0.0400 | hall_node_fraction=0.40 | hall_corr=0.250 | hall_mean=0.0060 | hall_std=0.0060 | hall_clamp=3.00 | nontarget_drift_weight=0.000
scenario 0 2018-07-10 AFG: target=-10.00% real=-1.47% hall=-3.65% diff=6.35%
scenario 1 2018-08-23 AFG: target=-10.00% real=-1.82% hall=-3.38% diff=6.62%
scenario 2 2021-05-07 AFG: target=-10.00% real=9.31% hall=6.69% diff=16.69%
scenario 3 2021-10-20 AFG: target=-10.00% real=7.28% hall=3.86% diff=13.86%
scenario 4 2025-05-15 AFG: target=-10.00% real=

## Step 6 (Optional): Goodness Backtest

In [14]:
if RUN_OPTIONAL_BACKTEST:
    run(
        f"{q(PYTHON_EXE)} scripts/goodness_backtest.py "
        f"--config {q(runtime_config)} "
        f"--out-csv {q(goodness_csv)} "
        f"--out-quantiles {q(goodness_quantiles)} "
        f"--out-plot {q(goodness_plot)} "
        f"--out-events {q(goodness_events)} "
        f"--out-strategy {q(goodness_strategy)} "
        f"--out-timeline {q(goodness_timeline)}"
    )
else:
    print("Skipping goodness_backtest (RUN_OPTIONAL_BACKTEST=False)")


/usr/bin/python3 scripts/goodness_backtest.py --config runs/experiments/e2e_runbook_20260213_205538/runtime_config.toml --out-csv runs/experiments/e2e_runbook_20260213_205538/diagnostics/goodness_backtest.csv --out-quantiles runs/experiments/e2e_runbook_20260213_205538/diagnostics/goodness_quantiles.csv --out-plot runs/experiments/e2e_runbook_20260213_205538/plots/goodness_scatter.png --out-events runs/experiments/e2e_runbook_20260213_205538/diagnostics/goodness_events.csv --out-strategy runs/experiments/e2e_runbook_20260213_205538/diagnostics/goodness_strategy_metrics.csv --out-timeline runs/experiments/e2e_runbook_20260213_205538/plots/goodness_timeline.png
backtest ticker: requested=AUTO effective=BWC source=auto_max_rows rows=10439
Wrote runs/experiments/e2e_runbook_20260213_205538/diagnostics/goodness_backtest.csv
Wrote runs/experiments/e2e_runbook_20260213_205538/diagnostics/goodness_quantiles.csv
Wrote runs/experiments/e2e_runbook_20260213_205538/plots/goodness_scatter.png
Wrot

## Final Artifact Inventory

In [15]:
artifacts = [
    runtime_config,
    graphs_pt,
    encoder_ckpt,
    critic_ckpt,
    model_ckpt,
    benchmark_csv,
    benchmark_plot,
    benchmark_bar,
    sweep_csv,
    sweep_summary,
    sweep_plot,
    sweep_pareto,
    scenario_csv,
    scenario_diag,
    stress_csv,
    stress_plot,
    calibration_json,
    calibration_by_ticker,
    goodness_csv,
    goodness_quantiles,
    goodness_plot,
    goodness_events,
    goodness_strategy,
    goodness_timeline,
]

rows = []
for p in artifacts:
    rows.append({
        "path": str(p),
        "exists": p.exists(),
        "bytes": p.stat().st_size if p.exists() and p.is_file() else None,
    })

artifact_df = pd.DataFrame(rows)
print(artifact_df.to_string(index=False))

missing = artifact_df.loc[~artifact_df["exists"], "path"].tolist()
if missing:
    print("\nMissing artifacts (may be expected if optional steps were disabled):")
    for p in missing:
        print("-", p)
else:
    print("\nAll tracked artifacts exist.")

                                                                                            path  exists     bytes
                                runs/experiments/e2e_runbook_20260213_205538/runtime_config.toml    True      7502
                                     runs/experiments/e2e_runbook_20260213_205538/data/graphs.pt    True 188988385
                                  runs/experiments/e2e_runbook_20260213_205538/models/encoder.pt    True    148437
                                   runs/experiments/e2e_runbook_20260213_205538/models/critic.pt    True     68947
                                  runs/experiments/e2e_runbook_20260213_205538/models/encoder.pt    True    148437
                              runs/experiments/e2e_runbook_20260213_205538/metrics/benchmark.csv    True      4814
                      runs/experiments/e2e_runbook_20260213_205538/plots/benchmark_speed_sep.png    True     35927
                            runs/experiments/e2e_runbook_20260213_205538/plots/b